In [1]:
# =========================================
# Cell 1 — Minimal installs (avoid scipy/sklearn conflicts)
# =========================================
!pip -q install --upgrade-strategy only-if-needed openai datasets huggingface_hub tqdm


In [2]:
# =========================================
# Cell 2 — Keys
# =========================================
import os, getpass

os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("DEEPSEEK_API_KEY: ").strip()
os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN (for gated FPB): ").strip()

assert os.environ["DEEPSEEK_API_KEY"]
assert os.environ["HF_TOKEN"]
print("✅ Keys set")


DEEPSEEK_API_KEY: ··········
HF_TOKEN (for gated FPB): ··········
✅ Keys set


In [3]:
# =========================================
# Cell 3 — Load FPB
# =========================================
from datasets import load_dataset
import os

DATASET_NAME = "TheFinAI/en-fpb"
SPLIT = "test"     # change if needed

ds = load_dataset(DATASET_NAME, split=SPLIT, token=os.environ["HF_TOKEN"])
print("✅ Loaded", DATASET_NAME, SPLIT, "N=", len(ds))
print("Columns:", ds.column_names)
print("Row0:", ds[0])


README.md:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

data/train-00000-of-00001-ab9a3b4799b095(…):   0%|          | 0.00/608k [00:00<?, ?B/s]

data/test-00000-of-00001-8bd1e21c671fb67(…):   0%|          | 0.00/188k [00:00<?, ?B/s]

data/valid-00000-of-00001-303e4ba2afe838(…):   0%|          | 0.00/154k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/776 [00:00<?, ? examples/s]

✅ Loaded TheFinAI/en-fpb test N= 970
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Row0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


In [7]:
# =========================================
# Cell 4 — DeepSeek Chat (single-thread, fast settings) + helpers
# =========================================
import re, time, json, random
from openai import OpenAI

DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MODEL_ID = "deepseek-chat"

client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url=DEEPSEEK_BASE_URL)

MAX_TOKENS = 8
TEMPERATURE = 0.0

LABELS_DEFAULT = ["negative", "neutral", "positive"]

def normalize_fpb(ex: dict):
    # text field (FPB variants)
    for k in ["text", "sentence", "content", "input"]:
        if k in ex:
            text = ex[k]
            break
    else:
        raise KeyError(f"No text field. Keys={list(ex.keys())}")

    # if already provides choices+gold
    if "choices" in ex and ("gold" in ex or "label" in ex):
        choices = list(ex["choices"])
        gold = int(ex.get("gold", ex.get("label")))
        return text, choices, gold

    # classic FPB: label is 0/1/2 or string
    choices = LABELS_DEFAULT
    lab = ex.get("label", ex.get("gold", ex.get("sentiment", None)))
    if lab is None:
        raise KeyError(f"No label/gold/sentiment. Keys={list(ex.keys())}")

    if isinstance(lab, str):
        lab_norm = lab.strip().lower()
        if lab_norm not in choices:
            raise ValueError(f"Unknown string label: {lab}")
        gold = choices.index(lab_norm)
    else:
        gold = int(lab)
        if gold < 0 or gold >= len(choices):
            raise ValueError(f"Label {gold} out of range for {choices}")

    return text, choices, gold

def make_messages(text: str, choices):
    # short prompt => faster + less tokens
    return [
        {"role": "system", "content": "Return ONLY one label from the options."},
        {"role": "user", "content": f"Text: {text}\nOptions: {', '.join(choices)}\nLabel:"},
    ]

def pick_label(raw: str, choices):
    if not raw:
        return None
    s = raw.strip().lower()

    for c in choices:
        if s == c.lower():
            return c
    for c in choices:
        if re.search(r"\b" + re.escape(c.lower()) + r"\b", s):
            return c
    return None

def deepseek_call(messages, max_retries=6):
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=messages,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                stream=False,
                timeout=60,
            )
            text_out = resp.choices[0].message.content if resp and resp.choices else None

            usage = getattr(resp, "usage", None)
            usage_dict = None
            if usage is not None:
                usage_dict = {
                    "prompt_tokens": getattr(usage, "prompt_tokens", None),
                    "completion_tokens": getattr(usage, "completion_tokens", None),
                    "total_tokens": getattr(usage, "total_tokens", None),
                }
            return text_out, {"usage": usage_dict}
        except Exception as e:
            last_err = str(e)
            # backoff (no sleep between successful calls; only on errors)
            time.sleep((2 ** attempt) * 0.6 + random.random() * 0.3)

    return None, {"error": last_err}


In [8]:
# =========================================
# Cell 5 — Evaluate (NO concurrency) + resume
# =========================================
import os, json, random
from tqdm import tqdm

SEED = 42
MAX_SAMPLES = None   # None = full split; set e.g. 200 to test
SHUFFLE = True

os.makedirs("text_responses", exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

run_tag = f"deepseekchat_{DATASET_NAME.replace('/','_')}_{SPLIT}"
OUT_PATH = f"text_responses/{run_tag}.jsonl"

random.seed(SEED)

# resume
done = set()
if os.path.exists(OUT_PATH):
    with open(OUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["row_index"])
            except:
                pass

idxs = list(range(len(ds)))
if SHUFFLE:
    random.shuffle(idxs)
if MAX_SAMPLES is not None:
    idxs = idxs[:MAX_SAMPLES]

remaining = [i for i in idxs if i not in done]
print("Already completed:", len(done), "| Remaining:", len(remaining))

with open(OUT_PATH, "a", encoding="utf-8") as f:
    for i in tqdm(remaining, desc="Evaluating"):
        ex = ds[i]
        text, choices, gold = normalize_fpb(ex)

        msgs = make_messages(text, choices)
        raw, meta = deepseek_call(msgs)

        pred_label = pick_label(raw, choices)
        pred_idx = choices.index(pred_label) if pred_label in choices else -1

        rec = {
            "row_index": i,
            "text": text,
            "choices": choices,
            "gold": gold,
            "model_response_raw": raw,
            "predicted_label": pred_label,
            "predicted_index": pred_idx,
            "correct": (pred_idx == gold),
            "meta": meta,
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("✅ Saved:", OUT_PATH)


Already completed: 970 | Remaining: 0


Evaluating: 0it [00:00, ?it/s]

✅ Saved: text_responses/deepseekchat_TheFinAI_en-fpb_test.jsonl


In [10]:
# =========================================
# Metrics output like sklearn.classification_report (NO sklearn)
# Requires: OUT_PATH pointing to your saved jsonl
# =========================================
import json
from math import isnan

# If OUT_PATH isn't defined in your notebook yet, set it here:
# OUT_PATH = "text_responses/deepseekchat_TheFinAI_en-fpb_test.jsonl"

rows = []
with open(OUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

assert len(rows) > 0, "No rows found in OUT_PATH"

# Use choices mapping from the file (assume consistent)
choices0 = rows[0].get("choices", ["negative", "neutral", "positive"])
for r in rows[:50]:
    if r.get("choices") != choices0:
        raise ValueError("Inconsistent 'choices' across rows. Please tell me and I’ll adapt the code.")

idx_to_label = {i: lab for i, lab in enumerate(choices0)}
labels = list(range(len(choices0)))

y_true = [int(r["gold"]) for r in rows]
y_pred = [int(r.get("predicted_index", -1)) for r in rows]  # -1 means invalid/unparsed

def safe_div(a, b):
    return a / b if b else 0.0

# Per-class counts
support = {c: 0 for c in labels}
tp = {c: 0 for c in labels}
fp = {c: 0 for c in labels}
fn = {c: 0 for c in labels}

for yt, yp in zip(y_true, y_pred):
    if yt in support:
        support[yt] += 1
    for c in labels:
        if yp == c and yt == c:
            tp[c] += 1
        elif yp == c and yt != c:
            fp[c] += 1
        elif yp != c and yt == c:
            fn[c] += 1

# Metrics per class
prec = {}
rec = {}
f1 = {}
for c in labels:
    prec[c] = safe_div(tp[c], tp[c] + fp[c])
    rec[c]  = safe_div(tp[c], tp[c] + fn[c])
    f1[c]   = safe_div(2 * prec[c] * rec[c], prec[c] + rec[c])

# Overall accuracy (count invalid preds as wrong)
correct = sum(int(yt == yp) for yt, yp in zip(y_true, y_pred))
acc = correct / len(y_true) if y_true else 0.0

# Macro / weighted averages
macro_p = sum(prec[c] for c in labels) / len(labels)
macro_r = sum(rec[c]  for c in labels) / len(labels)
macro_f = sum(f1[c]   for c in labels) / len(labels)

total_support = sum(support.values())
w_p = sum(prec[c] * support[c] for c in labels) / total_support if total_support else 0.0
w_r = sum(rec[c]  * support[c] for c in labels) / total_support if total_support else 0.0
w_f = sum(f1[c]   * support[c] for c in labels) / total_support if total_support else 0.0

# Pretty print (sklearn-like)
name_width = max(len(idx_to_label[c]) for c in labels + [labels[0]])
header = f"{'':>{name_width}}  {'precision':>9}  {'recall':>7}  {'f1-score':>8}  {'support':>7}"
print(f"Accuracy: {acc:.4f}\n")
print("Classification report:\n")
print(header)
print()

for c in labels:
    name = idx_to_label[c]
    print(f"{name:>{name_width}}  {prec[c]:9.4f}  {rec[c]:7.4f}  {f1[c]:8.4f}  {support[c]:7d}")

print()
print(f"{'accuracy':>{name_width}}  {'':>9}  {'':>7}  {acc:8.4f}  {total_support:7d}")
print(f"{'macro avg':>{name_width}}  {macro_p:9.4f}  {macro_r:7.4f}  {macro_f:8.4f}  {total_support:7d}")
print(f"{'weighted avg':>{name_width}}  {w_p:9.4f}  {w_r:7.4f}  {w_f:8.4f}  {total_support:7d}")


Accuracy: 0.7649

Classification report:

          precision   recall  f1-score  support

positive     0.7211   0.4946    0.5867      277
 neutral     0.7673   0.8856    0.8222      577
negative     0.8246   0.8103    0.8174      116

accuracy                        0.7649      970
macro avg     0.7710   0.7302    0.7421      970
weighted avg     0.7609   0.7649    0.7544      970
